# Part 4 · Notebook 07 — Order lifecycle, positions and reconciliation

**Sessions:** S11 (Order state machine & update streams) · S12 (Bracket, OCO, reconciliation) · [Lesson plan](../../docs/lessons/PART_04_BROKER_CONNECTIVITY.md) · graded labs in [`labs/part04/`](../../labs/part04/)

**You will:**
1. Apply a messy stream of order updates (duplicates, out of order, races) to a state machine.
2. Build positions from executions, not from order statuses.
3. Reconcile your open orders against the broker's after a crash.
4. Watch a bracket order's OCO children cancel each other.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing here connects to a broker: the account rows, bars, ticks and order events are synthetic, shaped like what `ib_async` and `alpaca-py` return.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p4lib.py is in notebooks/part04/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p4lib as p

p.use_course_style()

## 1. Update streams are messy

Order updates arrive over a network, from different broker subsystems. In one morning you can see: the same fill twice, a fill **before** the order's `ACCEPTED`, an `ACCEPTED` that arrives after `FILLED`, and a cancel that loses the race to a fill. Here are the events in **arrival order** (already mapped to canonical states).

In [ ]:
events = p.order_events()
pd.DataFrame(events).fillna("")

The naive way: the last status wins and every fill message adds up.

In [ ]:
naive = {}
for ev in events:
    o = naive.setdefault(ev["order_id"], {"state": "PENDING_NEW", "filled": Decimal(0)})
    if ev["type"] == "fill":
        o["filled"] += ev["qty"]
    else:
        o["state"] = ev["status"]
pd.DataFrame(naive).T

Every order is wrong: qf-101 shows 140 shares filled of 100, qf-102 is `ACCEPTED` although it is filled, qf-103 looks `CANCELLED` with 10 shares filled, and the rejected qf-104 looks live.

The fix: every order starts at `PENDING_NEW`; apply a status **only if `p.TRANSITIONS` allows it** from the current state (terminal states allow nothing); count each fill **once per `exec_id`**.

In [ ]:
pd.Series({k: ", ".join(sorted(v)) or "— terminal —" for k, v in p.TRANSITIONS.items()}, name="may move to").to_frame()

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def apply_events(events: list[dict]) -> dict[str, dict]:
    book, seen = {}, set()
    for ev in events:
        o = book.setdefault(ev["order_id"], {"state": "PENDING_NEW", "filled": Decimal(0), "notional": Decimal(0)})
        if ev["type"] == "fill":
            # ✍️ skip an exec_id already seen; otherwise remember it, add qty to filled and qty*price to notional
            ...
        elif ...:                                 # ✍️ only if p.TRANSITIONS allows o["state"] → ev["status"]
            o["state"] = ev["status"]
    return {k: {"state": o["state"], "filled": o["filled"],
                "avg_price": (o["notional"] / o["filled"]).quantize(Decimal("0.0001")) if o["filled"] else None}
            for k, o in book.items()}

mine = apply_events(events)
mine = p.check("apply_events", mine, p.apply_events(events))
pd.DataFrame(mine).T

Look at qf-103: we asked to cancel, but the fill won the race. `PENDING_CANCEL → FILLED` is a legal transition and the late `CANCELLED` is ignored. A strategy that assumed "I cancelled, so I'm flat" would now hold 10 shares it doesn't know about.

## 2. Positions come from executions

Order status says what happened to an *order*. Your *position* is the sum of executions: BUY adds, SELL subtracts. Leave out symbols that net to zero.

In [ ]:
fills = p.day_fills()
pd.DataFrame(fills)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def positions_from_fills(fills: list[dict]) -> dict[str, Decimal]:
    pos = {}
    for f in fills:
        pos[f["symbol"]] = ...                    # ✍️ previous position (0 if none) + qty for a BUY, − qty for a SELL
    return {s: q for s, q in pos.items() if q != 0}

mine = p.attempt(positions_from_fills, fills)
mine = p.check("positions_from_fills", mine, p.positions_from_fills(fills))
mine

## 3. Reconciliation after a crash

Your process crashed at 10:14 and restarted. Compare **your** open orders (client order id → remaining qty) with the **broker's**:

* `missing_at_broker`: in your book, not at the broker (filled or cancelled while you were down: check executions);
* `orphans`: at the broker with **your prefix** `qf-` but not in your book (sent just before the crash): cancel or adopt;
* `qty_mismatch`: on both sides with a different remaining qty (a fill you missed);
* `foreign`: at the broker without your prefix (placed by hand or by another system): **never touch**.

Every list sorted.

In [ ]:
D = Decimal
ours = {"qf-201": D("100"), "qf-202": D("50"), "qf-203": D("20"), "qf-204": D("10")}
broker = {"qf-201": D("100"), "qf-202": D("30"), "qf-205": D("15"), "qf-206": D("5"), "manual-77": D("200"), "TWS-3": D("1")}

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def reconcile(ours: dict, broker: dict, prefix: str = "qf-") -> dict[str, list[str]]:
    return {"missing_at_broker": sorted(set(ours) - set(broker)),
            "orphans": ...,                       # ✍️ broker ids with our prefix that we don't know
            "qty_mismatch": ...,                  # ✍️ ids on both sides with different qty
            "foreign": sorted(k for k in broker if not k.startswith(prefix))}

mine = reconcile(ours, broker)
mine = p.check("reconcile", mine, p.reconcile(ours, broker))
mine

## 4. A bracket order and its OCO children

A bracket is an entry with two exit children: a take-profit limit and a stop-loss. The children are **one-cancels-other**: when one fills, the broker cancels the other. Both brokers support this natively (IB: `ib.bracketOrder`; Alpaca: `order_class="bracket"`). Doing it client-side means a crash between the fill and your cancel leaves you with an unintended position.

In [ ]:
rng = np.random.default_rng(5)
path = 100 + np.cumsum(rng.normal(0, 0.12, 400))
entry, tp, sl = 100.0, 101.5, 99.0
state, log = "entry working", []
for i, px in enumerate(path):
    if state == "entry working" and px <= entry:
        state = "in position"; log.append((i, "entry filled at 100.00, TP and SL now working"))
    elif state == "in position" and (px >= tp or px <= sl):
        hit, other = ("take-profit", "stop-loss") if px >= tp else ("stop-loss", "take-profit")
        state = "flat"; log.append((i, f"{hit} filled at {px:.2f}; broker cancels the {other} (OCO)")); break
fig, ax = plt.subplots()
ax.plot(path[: log[-1][0] + 20])
for y, c, name in [(tp, p.PALETTE[2], "take-profit"), (entry, p.PALETTE[0], "entry"), (sl, p.PALETTE[7], "stop-loss")]:
    ax.axhline(y, color=c, ls="--", lw=1, label=name)
for i, _ in log:
    ax.plot(i, path[i], "o", color="black")
ax.set(xlabel="tick", ylabel="price", title="Bracket order"); ax.legend(); plt.show()
for i, msg in log:
    print(f"tick {i:3d}: {msg}")

## Wrap-up

* A state machine with legal transitions, and fills counted once per execution id, turn a messy stream into the truth.
* Positions come from executions; reconcile against the broker at startup and on a timer.
* Use broker-native brackets and OCO so exits survive your crash.
* Graded version: `labs/part04/week15_orders` (`OrderTracker`, `reconcile`) and the Clinic W3 event replay.